In [0]:
%pip install langchain-community
%pip install databricks-langchain
%pip install databricks-agents
dbutils.library.restartPython()

In [0]:
model = "databricks-meta-llama-3-3-70b-instruct"   # swap for another Databricks-served
                                                     # Foundation Model API endpoint if you prefer
vector_search_endpoint = "baby_reviews_vs_endpoint"
catalog = "rag_based_chatbot"
db = "default"
vector_search_index = "rag_reviews_index"

chain_config = {
    "llm_model_serving_endpoint_name": model,           # the foundation model we want to use
    "vector_search_endpoint_name": vector_search_endpoint,  # endpoint used for vector search
    "vector_search_index": f"{catalog}.{db}.{vector_search_index}",
    # WHY this prompt differs from the template's: it explicitly tells the model
    # the emoji sentiment is DATA it received, not something to invent — the same
    # anti-hallucination principle as the FL Studio example ("say I do not know"),
    # applied to the one thing unique to this project: sentiment must be reported
    # from the retrieved emoji, never guessed by the model itself.
    "llm_prompt_template": """You are a shopping assistant for baby products, helping customers
understand what other buyers experienced before they purchase. Every passage of context below
comes from a real customer review and includes: the product ID, the star rating, a sentiment
emoji (😍😊😐😞😡) that was computed from that review's own text — NOT your interpretation — and
an excerpt of the review itself.

Rules:
- Base your answer ONLY on the passages in the context. Do not invent product details or reviews.
- Always report sentiment using the emoji already attached to each passage — never assign your own.
- If the passages disagree (e.g. mixed sentiment), say so explicitly rather than picking one side.
- If the context does not contain enough information to answer, say "I don't have enough reviews
  to answer that" rather than guessing.

Context:
{context}""",
}

In [0]:
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

## Enable MLflow Tracing
mlflow.langchain.autolog()

## Load the chain's configuration
model_config = mlflow.models.ModelConfig(development_config=chain_config)

## Ensure Vector Search endpoint and index exist
from databricks.ai_search.client import VectorSearchClient as _AIVSC
import time as _time

_vsc = _AIVSC(disable_notice=True)
_vs_ep = model_config.get("vector_search_endpoint_name")
_vs_idx = model_config.get("vector_search_index")

# Create endpoint if missing
try:
    _vsc.get_endpoint(name=_vs_ep)
except Exception:
    print(f"[vs] creating endpoint '{_vs_ep}'...")
    _vsc.create_endpoint(name=_vs_ep)
    for _i in range(60):
        if _vsc.get_endpoint(name=_vs_ep).get("endpoint_status", {}).get("state") == "ONLINE":
            print(f"[vs] endpoint ready ({_i*10}s)")
            break
        _time.sleep(10)
    else:
        raise TimeoutError("VS endpoint not ready after 10 min")

# Create index if missing
try:
    _vsc.get_index(endpoint_name=_vs_ep, index_name=_vs_idx)
except Exception:
    print(f"[vs] creating index '{_vs_idx}' on rag_documents...")
    _vsc.create_delta_sync_index(
        endpoint_name=_vs_ep,
        index_name=_vs_idx,
        primary_key="doc_id",
        source_table_name="rag_based_chatbot.default.rag_documents",
        pipeline_type="TRIGGERED",
        embedding_source_column="document_text",
        embedding_model_endpoint_name="databricks-bge-large-en",
    )
    for _i in range(120):
        _st = _vsc.get_index(endpoint_name=_vs_ep, index_name=_vs_idx).describe() \
            .get("status", {}).get("detailed_state", "UNKNOWN")
        if _st in ("ONLINE", "ONLINE_NO_PENDING_UPDATE"):
            print(f"[vs] index ready ({_i*10}s)")
            break
        _time.sleep(10)
    else:
        raise TimeoutError("VS index not ready after 20 min")

## Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["doc_id", "asin", "overall", "summary_emoji", "document_text"],
).as_retriever(search_kwargs={"k": 5})   # k=5 to match the retrieval eval done earlier in the project

# WHY format_context changed: the template just dumped chunk text into the prompt.
# Here, each passage is prefixed with its emoji and rating BEFORE the text, so the
# LLM sees sentiment as structured, pre-computed metadata sitting right next to the
# claim it backs up — not something it has to infer from the prose itself.
def format_context(docs):
    chunk_contents = [
        f"Passage [{d.metadata.get('summary_emoji', '❓')} {d.metadata.get('overall', '?')}/5, "
        f"product {d.metadata.get('asin', 'unknown')}]: {d.page_content}\n"
        for d in docs
    ]
    return "".join(chunk_contents)

# Let's try our retriever chain:
relevant_docs = (
    vector_search_as_retriever | RunnableLambda(format_context) | StrOutputParser()
).invoke("Are baby monitors reliable and safe to use?")
displayHTML(relevant_docs)


In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages([
    ("system", model_config.get("llm_prompt_template")),  # instructions from the configuration
    ("user", "{question}"),                                 # user's question
])

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500},
)

# Let's try our prompt:
answer = (prompt | model | StrOutputParser()).invoke(
    {"question": "Are baby monitors reliable and safe to use?", "context": ""}
)
displayHTML(answer)


In [0]:
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
)

# Let's give it a try:
input_example = {
    "messages": [{"role": "user", "content": "Are baby monitors reliable and safe to use?"}]
}
answer = chain.invoke(input_example)
print(answer)

In [0]:
import yaml

rag_chain_config = {
    "databricks_resources": {
        "llm_endpoint_name": model_config.get("llm_model_serving_endpoint_name"),
        "vector_search_endpoint_name": model_config.get("vector_search_endpoint_name"),
    },
    "input_example": {
        "messages": [{"role": "user", "content": "Are baby monitors reliable and safe to use?"}]
    },
    "llm_config": {
        "llm_parameters": {"max_tokens": 1500, "temperature": 0.01},
        "llm_prompt_template": model_config.get("llm_prompt_template"),
        "llm_prompt_template_variables": ["context"],
    },
    "retriever_config": {
        # WHY chunk_template changed: same reasoning as format_context in Cell 3 —
        # the emoji/rating/product need to be part of what the retriever hands
        # back as "the passage," or they never reach the deployed model either.
        "chunk_template": "Passage [{summary_emoji} {overall}/5, product {asin}]: {document_text}\n",
        "parameters": {"k": 5},
        "schema": {
            "chunk_text": "document_text",
            "primary_key": "doc_id",
        },
        "vector_search_index": model_config.get("vector_search_index"),
    },
}

try:
    with open("rag_chain_config.yaml", "w") as f:
        yaml.dump(rag_chain_config, f)
except Exception:
    print("pass to work on build job")

model_config = mlflow.models.ModelConfig(development_config="rag_chain_config.yaml")


In [0]:
%%writefile chain.py

from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

## Enable MLflow Tracing
mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config="rag_chain_config.yaml")

# Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["doc_id", "asin", "overall", "summary_emoji", "document_text"],
).as_retriever(search_kwargs={"k": 5})  # number of reviews the retriever returns

# Enable the Review App and MLflow to properly track/display retrieved chunks
mlflow.models.set_retriever_schema(primary_key="doc_id", text_column="document_text")

# Method to format the docs returned by the retriever into the prompt,
# keeping the emoji and rating attached to each passage (see Cell 3's note)
def format_context(docs):
    chunk_contents = [
        f"Passage [{d.metadata.get('summary_emoji', '❓')} {d.metadata.get('overall', '?')}/5, "
        f"product {d.metadata.get('asin', 'unknown')}]: {d.page_content}\n"
        for d in docs
    ]
    return "".join(chunk_contents)

from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages([
    (   # System prompt contains the sentiment-aware instructions
        "system",
        """You are a shopping assistant for baby products, helping customers understand what
other buyers experienced before they purchase. Every passage of context below comes from a real
customer review and includes: the product ID, the star rating, a sentiment emoji (😍😊😐😞😡)
computed from that review's own text — NOT your interpretation — and an excerpt of the review.

Rules:
- Base your answer ONLY on the passages in the context. Do not invent product details or reviews.
- Always report sentiment using the emoji already attached to each passage — never assign your own.
- If the passages disagree (mixed sentiment), say so explicitly rather than picking one side.
- If the context is insufficient, say "I don't have enough reviews to answer that."

Context: {context}""",
    ),
    ("user", "{question}"),  # user's question
])

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500},
)

# Return the most recent user message as the question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
)

# Tell MLflow logging where to find your chain.
mlflow.models.set_model(model=chain)



In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from mlflow.models.signature import infer_signature
from mlflow.types.schema import ColSpec, Schema
import mlflow
import os

signature = infer_signature(input_example, answer)
signature.outputs = Schema([ColSpec(type="string")])

with mlflow.start_run(run_name="baby_reviews_sentiment_chatbot"):
    logged_chain_info = mlflow.langchain.log_model(
        lc_model=os.path.join(os.getcwd(), "chain.py"),
        model_config=chain_config,
        name="chain",
        input_example=input_example,
        signature=signature,
        resources=[
            DatabricksVectorSearchIndex(
                index_name=model_config.get("retriever_config")["vector_search_index"]
            ),
            DatabricksServingEndpoint(
                endpoint_name=model_config.get("databricks_resources")["llm_endpoint_name"]
            ),
        ],
    )


In [0]:
from databricks.agents import deploy
from databricks.agents.deployments import _create_endpoint_name, _create_new_endpoint_config
from databricks.sdk import WorkspaceClient
import mlflow
import time

# Register the logged chain as a UC model
MODEL_NAME_FQN = "rag_based_chatbot.default.baby_reviews_sentiment_chatbot"
uc_registered_model_info = mlflow.register_model(
    f"runs:/{logged_chain_info.run_id}/chain",
    MODEL_NAME_FQN,
)

w = WorkspaceClient()
endpoint_name = _create_endpoint_name(MODEL_NAME_FQN)

# Check if endpoint already exists
try:
    w.serving_endpoints.get(endpoint_name)
    endpoint_exists = True
    print(f"[deploy] endpoint '{endpoint_name}' already exists")
except Exception:
    endpoint_exists = False
    print(f"[deploy] endpoint '{endpoint_name}' does not exist yet")

# Pre-create endpoint if it doesn't exist (workaround for agents.deploy stuck issue)
if not endpoint_exists:
    print(f"[deploy] pre-creating endpoint '{endpoint_name}' without inference tables...")
    w.serving_endpoints.create(
        name=endpoint_name,
        config=_create_new_endpoint_config(
            MODEL_NAME_FQN,
            int(uc_registered_model_info.version),
            endpoint_name,
            scale_to_zero=True,
            workload_size="Small",
        ),
    )

# Wait for endpoint to be ready
for i in range(180):
    ep = w.serving_endpoints.get(endpoint_name)
    ready = ep.state.ready.name if ep.state and ep.state.ready else "UNKNOWN"
    config_update = ep.state.config_update.name if ep.state and ep.state.config_update else "UNKNOWN"
    if ready == "READY":
        print(f"[deploy] endpoint ready after {i*10}s")
        break
    if config_update == "UPDATE_FAILED":
        raise RuntimeError(f"Update FAILED — check Serving > {endpoint_name} > Events")
    if i % 6 == 0:
        print(f"[deploy] waiting... ({config_update}/{ready}, {i*10}s elapsed)")
    time.sleep(10)
else:
    raise TimeoutError(f"Endpoint not ready after 30 min — check Serving > {endpoint_name}")

# Deploy the agent
deployment_info = deploy(
    MODEL_NAME_FQN,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True,
)
print(f"[deploy] deployed: {deployment_info}")

In [0]:
from databricks.agents.deployments import _create_endpoint_name
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
endpoint_name = _create_endpoint_name(MODEL_NAME_FQN)

# ---- 1. Delete the wedged endpoint — it cannot recover on its own,
# it's been stuck on the same conflicting update for 11+ minutes ----
try:
    w.serving_endpoints.delete(endpoint_name)
    print(f"[deploy] deleted stuck endpoint '{endpoint_name}'")
    time.sleep(15)  # give the deletion a moment to fully complete
except Exception as e:
    print(f"[deploy] no endpoint to delete or already gone: {e}")

# ---- 2. Recreate it via the plain SDK, NOT agents.deploy() ----
from databricks.agents.deployments import _create_new_endpoint_config

w.serving_endpoints.create(
    name=endpoint_name,
    config=_create_new_endpoint_config(
        MODEL_NAME_FQN,
        int(uc_registered_model_info.version),
        endpoint_name,
        scale_to_zero=True,
        workload_size="Small",
    ),
)

# ---- 3. Wait, with FAILURE detection this time ----
for i in range(180):
    ep = w.serving_endpoints.get(endpoint_name)
    ready = ep.state.ready.name if ep.state and ep.state.ready else "UNKNOWN"
    config_update = ep.state.config_update.name if ep.state and ep.state.config_update else "UNKNOWN"
    if ready == "READY":
        print(f"[deploy] ready after {i*10}s")
        break
    if config_update == "UPDATE_FAILED":
        raise RuntimeError("Update FAILED — check Serving > Events in the UI for the real error.")
    if i % 6 == 0:
        print(f"[deploy] waiting... ({config_update}/{ready}, {i*10}s elapsed)")
    time.sleep(10)
else:
    raise TimeoutError(f"Still not ready after 30 min — check Serving > {endpoint_name} > Events.")

# ---- 4. Get your URL — no agents.deploy() involved at all ----
endpoint_url = f"{w.config.host}/serving-endpoints/{endpoint_name}/invocations"
print(f"[deploy] chatbot URL: {endpoint_url}")

response = w.serving_endpoints.query(
    name=endpoint_name,
    inputs={"messages": [{"role": "user", "content": "Are baby monitors reliable and safe to use?"}]},
)
print(response)